# Быстрый запуск ruT5

Готовый checkpoint последнего эксперимента. Выполняй ячейки сверху вниз из корня репозитория после `pip install -r requirements.txt`.

Модель скачивается из приватного GitHub Release только при первом запуске. Для доступа нужен `gh auth login` или `GH_TOKEN`. Обучение и полный расчёт метрик здесь не запускаются.

In [ ]:
import os
import re

import torch
from transformers import T5Tokenizer, MT5ForConditionalGeneration

from download_model import download_model

MODEL_DIR = download_model()
MAX_LENGTH = 1536
device = torch.device(
    os.environ.get("RUT5_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
)
print("Устройство:", device)

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
model = MT5ForConditionalGeneration.from_pretrained(
    str(MODEL_DIR),
    dtype=torch.float32,
    local_files_only=True,
).to(device)
model.eval()
print("Загружена модель:", MODEL_DIR)

In [ ]:
@torch.inference_mode()
def translate(text):
    model.eval()
    text = text.replace("\xa0", " ")
    text = re.sub(r"(?m)^[ \t]*\{\d+\}[ \t]*$", "", text).strip()
    if not text:
        return ""
    inputs = tokenizer(text, return_tensors="pt", truncation=False).to(device)
    if inputs["input_ids"].shape[1] > MAX_LENGTH:
        raise ValueError(f"Текст длиннее {MAX_LENGTH} токенов.")
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_LENGTH,
        num_beams=4,
        do_sample=False,
        no_repeat_ngram_size=3,
        repetition_penalty=1.1,
        use_cache=True,
    )
    if output[0, -1].item() != tokenizer.eos_token_id:
        print("Достигнут лимит генерации; ответ может быть неполным.")
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
examples = [
    "Это правда.",
    "Это великолепный результат.",
    "Он опять начал хвастаться.",
]

for text in examples:
    print("Оригинал:", text)
    print("Перевод:", translate(text))
    print()

Меняй список `examples` или вызывай `translate("свой текст")`. Повторно загружать модель между вызовами не нужно.

На CPU длинные тексты и поиск с четырьмя лучами медленнее коротких примеров. Модель экспериментальная: возможны копирование оригинала, ошибки согласования и искажение смысла. Она не применяет JSON-словарь как жёсткие правила.